# 03 Medical Image Classification
### จำแนกภาพถ่ายทางการแพทย์ด้วย Transfer Learning

ข้อมูลภาพการแพทย์มักมีจำกัด เราจึงใช้ **transfer learning**, เริ่มจากโมเดล ที่ pretrain มาแล้ว แล้ว fine-tune กับข้อมูลของเรา

> 🖼️ อ่านพื้นฐานที่ [Medical Imaging](../curriculum/health/medical-imaging.html)

## 1. เตรียมสภาพแวดล้อม

บน Google Colab เปิด GPU: **Runtime → Change runtime type → T4 GPU**

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'ใช้ device: {device}')

ใช้ device: cuda


## 2. เตรียมการแปลงภาพ (Preprocessing)

ภาพต้อง resize และ normalize ให้ตรงกับที่โมเดล pretrain ใช้

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
print('Pipeline การแปลงภาพพร้อมแล้ว')

Pipeline การแปลงภาพพร้อมแล้ว


## 3. โหลดโมเดล pretrain และปรับชั้นสุดท้าย

เราใช้ ResNet-18 แล้วเปลี่ยน classifier ให้เป็น 2 คลาส (เช่น ปกติ / ผิดปกติ), freeze ชั้นต้น ฝึกเฉพาะชั้นใหม่

In [ ]:
model = models.resnet18(weights='IMAGENET1K_V1')
for p in model.parameters():
    p.requires_grad = False          # freeze ชั้นต้น

model.fc = nn.Linear(model.fc.in_features, 2)   # 2 คลาส
model = model.to(device)
print('โมเดลพร้อม fine-tune (ฝึกเฉพาะชั้นสุดท้าย)')

โมเดลพร้อม fine-tune (ฝึกเฉพาะชั้นสุดท้าย)


## 4. โครงการ train loop (ตัวอย่าง)

เมื่อมี DataLoader ของภาพจริง loop การ train จะเป็นดังนี้

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

def train_one_epoch(loader):
    model.train()
    total = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(images)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

print('นิยาม train loop เรียบร้อย, พร้อมป้อน DataLoader ของคุณ')

นิยาม train loop เรียบร้อย, พร้อมป้อน DataLoader ของคุณ


> ⚠️ **เครื่องมือสาธิต ≠ เครื่องมือทางการแพทย์**
> โมเดลใน notebook นี้เพื่อการศึกษา การใช้กับผู้ป่วยจริงต้องผ่าน clinical validation และการกำกับดูแลตามที่เรียนใน AI Ethics

## สรุปและก้าวต่อไป

- เราใช้ transfer learning ปรับ ResNet-18 ให้จำแนกภาพการแพทย์
- การ freeze ชั้นต้นช่วยให้ train ได้แม้ข้อมูลน้อย

**ลองต่อ:** unfreeze บางชั้นแล้ว fine-tune ต่อ (lr ต่ำ ๆ) เพื่อเพิ่มความแม่น แล้วนำไป deploy ตาม [Week 8, Deployment](../curriculum/capstone/deployment.html)